# LiDAR/Wave Sweep Lab — upload A + B, then run

This notebook is designed for the simple Colab flow:

**A = engine file**: `lidar_lenses_wave_v056.py` or newer  
**B = sweep spreadsheet**: `lidar_wave_sweep_plan_v056.xlsx` or compatible

Run the first cell, upload both files when prompted, then run the rest. The notebook imports the engine directly from the uploaded `.py` path, so the filename/cwd import problem goes away.


In [ ]:
# 1) Upload A + B and import the engine directly from the uploaded .py file.
from pathlib import Path
import sys, os, json, zipfile, shutil, importlib.util
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw

WORK_DIR = Path("/content/lidar_upload_ab_lab")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

print("Working folder:", WORK_DIR)
print("\nUpload two files:")
print("  A) lidar_lenses_wave_v060.py  (or newer engine .py)")
print("  B) lidar_wave_sweep_plan_v056.xlsx  (or your edited sweep plan)")
print()

try:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        path = WORK_DIR / name
        path.write_bytes(data)
except Exception as e:
    print("Colab upload widget was not available or was skipped.")
    print("Put the .py and .xlsx files in:", WORK_DIR)
    print("Details:", repr(e))

# Find engine and spreadsheet.
py_files = sorted(WORK_DIR.glob("*.py"), key=lambda p: p.stat().st_mtime, reverse=True)
xlsx_files = sorted(WORK_DIR.glob("*.xlsx"), key=lambda p: p.stat().st_mtime, reverse=True)

if not py_files:
    raise FileNotFoundError(f"No engine .py file found in {WORK_DIR}. Upload lidar_lenses_wave_v060.py or newer.")
if not xlsx_files:
    print("No .xlsx sweep plan found. A small default plan will be created later.")
    plan_path = None
else:
    plan_path = xlsx_files[0]

engine_path = py_files[0]
print("Engine file:", engine_path.name)
print("Plan file:", plan_path.name if plan_path else "(will create default plan)")

# Import from file path so sys.path/current directory does not matter.
spec = importlib.util.spec_from_file_location("llw_uploaded_engine", str(engine_path))
llw = importlib.util.module_from_spec(spec)
spec.loader.exec_module(llw)

print("\nLoaded engine from:", engine_path)
doc = (llw.__doc__ or "").strip().splitlines()
if doc:
    print("Engine doc:", doc[0])

OUT_DIR = WORK_DIR / "sweep_outputs_upload_ab"
OUT_DIR.mkdir(exist_ok=True)
print("Output folder:", OUT_DIR)


## Scene builders

The engine itself has a built-in cabin scene. The notebook adds three quick stress scenes so the spreadsheet can refer to:

`cabin_demo`, `mini_saloon`, `occluder_gate`, `material_targets`


In [ ]:
# 2) Quick procedural scenes.
def _R(rx=0, ry=0, rz=0):
    return llw.make_rotation_matrix(rx, ry, rz)

def _add(prims, shape, center, half_extents, color, piece_type, rotation=None, transparency=0.0, piece_id=None):
    if rotation is None:
        rotation = _R()
    if piece_id is None:
        piece_id = len(prims)
    prims.append(llw.Primitive(
        shape=shape,
        center=np.array(center, dtype=np.float64),
        half_extents=np.array(half_extents, dtype=np.float64),
        rotation_matrix=rotation.T,
        inv_rotation_matrix=rotation,
        color=tuple(color),
        piece_id=int(piece_id),
        piece_type=str(piece_type),
        transparency=float(transparency),
    ))

def build_mini_saloon():
    prims = []
    # floor and walls
    _add(prims, "box", [0, -0.05, 0], [7.0, 0.05, 6.0], (0.45, 0.31, 0.18), "wood_floor")
    _add(prims, "box", [0, 1.6, 3.05], [7.0, 1.6, 0.12], (0.50, 0.32, 0.18), "wood_wall")
    _add(prims, "box", [-3.55, 1.6, 0], [0.12, 1.6, 6.0], (0.48, 0.28, 0.16), "wood_wall")
    _add(prims, "box", [3.55, 1.6, 0], [0.12, 1.6, 6.0], (0.48, 0.28, 0.16), "wood_wall")
    # bar counter and back shelf
    _add(prims, "box", [0.0, 0.65, 2.0], [4.6, 0.65, 0.45], (0.35, 0.20, 0.10), "bar_counter")
    _add(prims, "box", [0.0, 1.65, 2.85], [4.8, 0.08, 0.15], (0.30, 0.18, 0.09), "shelf")
    for x in np.linspace(-2.0, 2.0, 7):
        _add(prims, "cylinder", [x, 1.95, 2.67], [0.06, 0.22, 0.0], (0.1, 0.55, 0.35), "bottle", rotation=_R())
    # tables/chairs
    for cx, cz in [(-1.8, -0.6), (1.5, -0.7), (0.0, -2.1)]:
        _add(prims, "cylinder", [cx, 0.52, cz], [0.45, 0.06, 0.0], (0.34, 0.18, 0.08), "table_top")
        _add(prims, "cylinder", [cx, 0.25, cz], [0.08, 0.25, 0.0], (0.25, 0.14, 0.06), "table_leg")
        for dx, dz, rot in [(0.75,0,0),(-0.75,0,0),(0,0.75,90),(0,-0.75,90)]:
            _add(prims, "box", [cx+dx, 0.28, cz+dz], [0.32,0.28,0.32], (0.38,0.20,0.08), "chair")
            _add(prims, "box", [cx+dx, 0.72, cz+dz], [0.35,0.55,0.08], (0.30,0.15,0.06), "chair_back", rotation=_R(0,rot,0))
    # stools
    for x in np.linspace(-2.0, 2.0, 5):
        _add(prims, "cylinder", [x, 0.48, 1.25], [0.18, 0.07, 0.0], (0.30,0.14,0.06), "stool")
        _add(prims, "cylinder", [x, 0.24, 1.25], [0.05, 0.24, 0.0], (0.18,0.10,0.05), "stool_leg")
    return prims

def build_occluder_gate():
    prims = []
    _add(prims, "box", [0, -0.05, 0], [9, 0.05, 7], (0.32,0.30,0.22), "ground")
    _add(prims, "box", [0, 1.2, 2.8], [7.5, 1.2, 0.12], (0.55,0.55,0.52), "back_wall")
    # fence/gate
    for x in np.linspace(-3, 3, 9):
        _add(prims, "cylinder", [x, 0.8, 0.2], [0.045, 0.8, 0.0], (0.35,0.22,0.10), "fence", transparency=0.55)
    for y in [0.55, 1.15]:
        _add(prims, "box", [0, y, 0.2], [6.4, 0.04, 0.06], (0.32,0.20,0.09), "fence", transparency=0.55)
    # foliage blobs
    for x,z,r in [(-2.8,1.2,0.9),(-1.5,1.5,0.75),(1.8,1.3,0.85),(3.0,1.0,0.7)]:
        _add(prims, "sphere", [x, 1.4, z], [r,r,r], (0.16,0.50,0.18), "canopy", transparency=0.65)
        _add(prims, "cylinder", [x, 0.55, z], [0.10,0.55,0.0], (0.28,0.16,0.08), "trunk")
    # hard boxes behind occluder
    for x in [-1.2, 1.2]:
        _add(prims, "box", [x, 0.5, 2.0], [0.7,0.5,0.5], (0.65,0.62,0.55), "stone")
    return prims

def build_material_targets():
    prims = []
    _add(prims, "box", [0, -0.05, 0], [9, 0.05, 5], (0.30,0.30,0.28), "ground")
    mats = [
        ("wood",   (-3.2,0.6,0), (0.45,0.25,0.12), "cylinder"),
        ("metal",  (-1.9,0.6,0), (0.72,0.74,0.76), "box"),
        ("glass",  (-0.6,0.6,0), (0.60,0.80,0.95), "box"),
        ("cloth",  (0.7,0.6,0),  (0.65,0.30,0.25), "box"),
        ("stone",  (2.0,0.6,0),  (0.45,0.45,0.46), "sphere"),
        ("foliage",(3.3,0.75,0), (0.18,0.52,0.20), "sphere"),
    ]
    for name, center, color, shape in mats:
        if shape == "sphere":
            _add(prims, shape, center, [0.55,0.55,0.55], color, name, transparency=0.50 if name=="foliage" else 0.0)
        elif shape == "cylinder":
            _add(prims, shape, center, [0.35,0.6,0.0], color, name)
        else:
            _add(prims, shape, center, [0.9,1.0,0.7], color, name)
    return prims

def build_scene(scene_name):
    s = str(scene_name).strip()
    if s == "cabin_demo":
        return llw._build_demo_scene()
    if s == "mini_saloon":
        return build_mini_saloon()
    if s == "occluder_gate":
        return build_occluder_gate()
    if s == "material_targets":
        return build_material_targets()
    raise ValueError(f"Unknown scene {scene_name!r}. Valid: cabin_demo, mini_saloon, occluder_gate, material_targets")

print("Scene builders ready.")


## Read the spreadsheet

The spreadsheet may have a sheet named `SweepPlan`; otherwise the first sheet is used. Required columns are only `scene` and `preset`. Other columns override preset settings.


In [ ]:
# 3) Read or create the sweep plan.
DEFAULT_PLAN = pd.DataFrame([
    dict(scene="mini_saloon", preset="overhead_layout", width=640, height=640, rays_per_pixel=1, stack=1, carrier_strength=1.0, edge_anti_min=0.30, adaptive_edge_percentile=90, carrier_mode="ensemble", include_polarization=False, seed=42, notes="layout"),
    dict(scene="mini_saloon", preset="indoor_structure", width=480, height=320, rays_per_pixel=8, stack=4, carrier_strength=1.0, edge_anti_min=0.35, adaptive_edge_percentile=92, carrier_mode="ensemble", include_polarization=False, seed=42, notes="indoor anti channels"),
    dict(scene="occluder_gate", preset="outdoor_occlusion", width=480, height=320, rays_per_pixel=8, stack=4, carrier_strength=1.0, edge_anti_min=0.30, adaptive_edge_percentile=90, carrier_mode="ensemble", include_polarization=False, seed=42, notes="partial occluders"),
    dict(scene="material_targets", preset="material_scan", width=480, height=320, rays_per_pixel=8, stack=4, carrier_strength=1.0, edge_anti_min=0.30, adaptive_edge_percentile=90, carrier_mode="ensemble", include_polarization=True, seed=42, notes="materials"),
    dict(scene="cabin_demo", preset="full_diagnostic", width=480, height=320, rays_per_pixel=8, stack=4, carrier_strength=1.0, edge_anti_min=0.30, adaptive_edge_percentile=90, carrier_mode="ensemble", include_polarization=True, seed=42, notes="full diagnostics"),
])

if plan_path is None:
    plan_df = DEFAULT_PLAN.copy()
    plan_path = WORK_DIR / "default_sweep_plan_created.xlsx"
    plan_df.to_excel(plan_path, index=False, sheet_name="SweepPlan")
    print("Created default plan:", plan_path)
else:
    xl = pd.ExcelFile(plan_path)
    sheet = "SweepPlan" if "SweepPlan" in xl.sheet_names else xl.sheet_names[0]
    plan_df = pd.read_excel(plan_path, sheet_name=sheet)
    print("Loaded sheet:", sheet)

# Clean up columns.
plan_df.columns = [str(c).strip() for c in plan_df.columns]
if "enabled" in plan_df.columns:
    plan_df = plan_df[plan_df["enabled"].fillna(True).astype(bool)].copy()

required = {"scene", "preset"}
missing = required - set(plan_df.columns)
if missing:
    raise ValueError(f"Sweep plan missing required columns: {missing}")

print("Runs:", len(plan_df))
display(plan_df.head(20))


## Run the sweep

This cell runs each row, saves contact sheets + diagnostics + channels, and writes CSV/XLSX metrics.

In [ ]:
# 4) Run sweep and collect metrics.
import time, math

def _clean_value(v):
    if pd.isna(v):
        return None
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        # preserve floats, but convert whole floats to int for width/height/etc later
        return float(v)
    return v

OVERRIDE_COLUMNS = [
    "width", "height", "rays_per_pixel", "stack", "carrier_strength",
    "edge_anti_min", "adaptive_edge_percentile", "carrier_mode",
    "include_polarization", "attenuation_per_meter", "depth_wavelength",
    "sampling_mode", "auto_frame", "fov_deg"
]

def run_one(row, run_index):
    scene_name = str(row["scene"]).strip()
    preset_name = str(row["preset"]).strip()
    seed = int(row.get("seed", 42) if not pd.isna(row.get("seed", 42)) else 42)

    overrides = {}
    for col in OVERRIDE_COLUMNS:
        if col in row.index:
            val = _clean_value(row[col])
            if val is None:
                continue
            if col in ["width", "height", "rays_per_pixel", "stack"]:
                val = int(val)
            if col in ["include_polarization", "auto_frame"]:
                if isinstance(val, str):
                    val = val.strip().lower() in ("1", "true", "yes", "y", "on")
                else:
                    val = bool(val)
            overrides[col] = val

    prims = build_scene(scene_name)
    run_name = f"{run_index:03d}_{scene_name}_{preset_name}"
    run_dir = OUT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    t0 = time.time()
    result = llw.run_sensor_preset(
        prims,
        preset_name=preset_name,
        scene_name=scene_name,
        out_dir=str(run_dir),
        seed=seed,
        **overrides
    )
    elapsed = time.time() - t0

    diag = result["diagnostics"]
    stats = diag.get("depth_stats", {})
    counts = diag.get("classification_counts") or {}
    channels = result.get("channels")

    hit_px = sum(v for k, v in counts.items() if k != "sky") if counts else 0
    def frac(label):
        return float(counts.get(label, 0) / max(hit_px, 1))

    row_out = {
        "run_index": run_index,
        "scene": scene_name,
        "preset": preset_name,
        "notes": row.get("notes", ""),
        "runtime_seconds": round(float(elapsed), 3),
        "coverage": float(stats.get("coverage", 0.0) or 0.0),
        "depth_span_p05_p95": float(stats.get("depth_span_p05_p95", 0.0) or 0.0),
        "depth_p50": stats.get("depth_p50"),
        "width": diag.get("width"),
        "height": diag.get("height"),
        "rays_per_pixel": diag.get("rays_per_pixel"),
        "stacked_bursts": diag.get("stacked_bursts"),
        "carrier_mode": diag.get("carrier_mode"),
        "edge_anti_min": diag.get("edge_anti_min"),
        "adaptive_edge_percentile": diag.get("adaptive_edge_percentile"),
        "geom_edge_frac": frac("geom_edge"),
        "partial_occluder_frac": frac("partial_occluder"),
        "foliage_frac": frac("foliage"),
        "solid_surface_frac": frac("solid_surface"),
        "hard_smooth_frac": frac("hard_smooth"),
        "soft_material_frac": frac("soft_material"),
        "uncertain_frac": frac("uncertain"),
        "contact_sheet": result["paths"].get("contact_sheet"),
        "diagnostics_path": result["paths"].get("diagnostics"),
        "channels_path": result["paths"].get("channels"),
    }

    if channels is not None:
        hits = channels["hit_count"] > 0
        for key in ["light_anti", "sound_anti", "light_coh", "sound_coh", "acoustic_intensity", "ultrasonic_intensity", "acoustic_softness", "depth_variance"]:
            if key in channels and hits.any():
                arr = channels[key][hits]
                row_out[f"{key}_mean"] = float(np.mean(arr))
                row_out[f"{key}_p90"] = float(np.percentile(arr, 90))
                row_out[f"{key}_p95"] = float(np.percentile(arr, 95))
            else:
                row_out[f"{key}_mean"] = np.nan
                row_out[f"{key}_p90"] = np.nan
                row_out[f"{key}_p95"] = np.nan

    # Heuristic score: rewards coverage, depth span, useful anti response, and non-chaotic labels.
    anti = np.nanmean([row_out.get("light_anti_p95", np.nan), row_out.get("sound_anti_p95", np.nan)])
    if np.isnan(anti):
        anti = 0.0
    cov_score = min(row_out["coverage"] / 0.65, 1.0)
    span_score = min(row_out["depth_span_p05_p95"] / 8.0, 1.0)
    edge_penalty = max(0.0, row_out["geom_edge_frac"] - 0.18) * 0.8
    uncertain_penalty = max(0.0, row_out["uncertain_frac"] - 0.25) * 0.4
    row_out["score"] = round(float(0.35*cov_score + 0.20*span_score + 0.35*anti - edge_penalty - uncertain_penalty), 4)

    return row_out

metrics = []
for i, (_, row) in enumerate(plan_df.iterrows(), start=1):
    print(f"\n[{i}/{len(plan_df)}] {row['scene']} / {row['preset']}")
    try:
        m = run_one(row, i)
        metrics.append(m)
        print("  coverage:", round(m["coverage"], 3), "span:", round(m["depth_span_p05_p95"], 2), "score:", m["score"])
    except Exception as e:
        print("  ERROR:", repr(e))
        metrics.append({
            "run_index": i,
            "scene": row.get("scene"),
            "preset": row.get("preset"),
            "error": repr(e),
            "score": -999,
        })

metrics_df = pd.DataFrame(metrics)
metrics_csv = OUT_DIR / "sweep_metrics.csv"
metrics_xlsx = OUT_DIR / "sweep_metrics.xlsx"
metrics_df.to_csv(metrics_csv, index=False)
metrics_df.to_excel(metrics_xlsx, index=False)

print("\nSaved:")
print(metrics_csv)
print(metrics_xlsx)

display(metrics_df.sort_values("score", ascending=False).head(20))


## Make a top-contact-sheet overview and ZIP everything

In [ ]:
# 5) Top 2–5 overview + ZIP download.
from PIL import Image, ImageDraw

def make_top_overview(metrics_df, top_n=5, thumb_w=420):
    valid = metrics_df.copy()
    valid = valid[valid.get("contact_sheet").notna()] if "contact_sheet" in valid else valid.iloc[0:0]
    valid = valid.sort_values("score", ascending=False).head(top_n)
    images, labels = [], []
    for _, r in valid.iterrows():
        p = Path(str(r["contact_sheet"]))
        if not p.exists():
            continue
        im = Image.open(p).convert("RGB")
        scale = thumb_w / im.width
        im = im.resize((thumb_w, max(1, int(im.height * scale))))
        label = f"#{int(r['run_index'])} {r['scene']} / {r['preset']} | score {r['score']:.3f} | cov {r.get('coverage',0):.2f}"
        images.append(im)
        labels.append(label)

    if not images:
        return None

    pad, label_h = 12, 28
    W = thumb_w + 2*pad
    H = pad + sum(im.height + label_h + pad for im in images)
    out = Image.new("RGB", (W, H), (10, 10, 14))
    draw = ImageDraw.Draw(out)
    y = pad
    for im, lab in zip(images, labels):
        draw.text((pad, y), lab, fill=(225, 230, 230))
        y += label_h
        out.paste(im, (pad, y))
        y += im.height + pad
    return out

top_n = 5
overview = make_top_overview(metrics_df, top_n=top_n)
overview_path = OUT_DIR / "top_contact_sheets.png"
if overview:
    overview.save(overview_path)
    display(overview)
    print("Saved overview:", overview_path)
else:
    print("No overview created; no successful contact sheets found.")

zip_path = WORK_DIR / "lidar_wave_sweep_outputs_upload_ab.zip"
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path).replace(".zip",""), "zip", OUT_DIR)
print("Zipped outputs:", zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("Download manually from:", zip_path)
